# Customer Review Sentiment Classification (TF-IDF)

This notebook preprocesses review text, creates TF-IDF features (`max_features=5000`), and benchmarks Multinomial Naive Bayes against Logistic Regression. The reusable implementation is in `sentiment_classifier.py`; the notebook uses the same functions as the command-line runner.

## Dataset and scope

The local CSV contains only 10 reviews: 4 positive, 5 negative, and 1 neutral. This exercise is binary, so the neutral row is kept in the source data and reported, but excluded from training and scoring. With so few examples, a single prediction can change the holdout accuracy substantially; results are illustrative only, not reliable performance estimates. A much larger, representative labeled dataset is needed for meaningful evaluation.

In [ ]:
from pathlib import Path
import sys

from IPython.display import display

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_DIR = next(
    (path for path in candidates if (path / "sentiment_classifier.py").is_file()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError("Could not find review/sentiment_classifier.py from the current directory.")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import sentiment_classifier as sentiment

## Load and inspect

Preprocessing removes HTML, lowercases and tokenizes words, removes English stopwords while retaining negations, and lemmatizes tokens with NLTK WordNet. The TF-IDF vectorizer is fitted only on each model's training partition to avoid test-data leakage.

In [ ]:
data = sentiment.load_dataset(PROJECT_DIR / "data.csv")
texts, labels, neutral_count = sentiment.prepare_binary_dataset(data)
print(f"Total reviews: {len(data)} | Binary reviews: {len(texts)} | Neutral excluded: {neutral_count}")
display(data["sentiment"].value_counts().rename_axis("sentiment").to_frame("count"))
display(data.head())

## Train and compare models

A fixed-seed, stratified 75/25 holdout is used. Metrics include accuracy and positive-class precision, recall, and F1. The split is tiny, so do not interpret close scores as evidence that one model is generally better.

In [ ]:
run = sentiment.run_benchmark(PROJECT_DIR / "data.csv", PROJECT_DIR / "outputs")
display(run["summary"].style.format({
    "accuracy": "{:.3f}",
    "precision_positive": "{:.3f}",
    "recall_positive": "{:.3f}",
    "f1_positive": "{:.3f}",
}))
print("Saved benchmark metadata and reports to:", PROJECT_DIR / "outputs")

## Try custom reviews

`predict_sentiment()` returns a predicted class and the model's probability for that class. Confidence is not a guarantee of correctness, especially for this tiny training set.

In [ ]:
naive_bayes = run["results"]["Multinomial Naive Bayes"]["estimator"]

def predict_live_review(text: str) -> dict[str, str | float]:
    """Classify one custom positive/negative customer review."""
    return sentiment.predict_sentiment(text, naive_bayes)

examples = [
    "I love this product; it works perfectly and feels durable.",
    "The item broke immediately and customer service was terrible.",
]
for review in examples:
    print(review, "->", predict_live_review(review))